# Análise Comparativa de Modelos — AlignFit

## Modelos comparados
| Modelo | Tipo | Característica principal |
|--------|------|-------------------------|
| **XGBoost** | Ensemble (Gradient Boosting) | Árvores sequenciais com regularização |
| **KNN** | Instance-based | Classificação por vizinhos mais próximos |
| **SVM** | Kernel-based | Separação por hiperplano em espaço de alta dimensão |
| **Decision Tree** | Árvore de decisão | Particionamento recursivo por atributos |

## Métricas avaliadas
- Accuracy, Precision, Recall, F1-Score
- Tempo de treinamento e inferência
- Cross-validation (5-fold stratified)

In [ ]:
# =========================================================
# IMPORTS
# =========================================================

import pandas as pd
import numpy as np
import time
import warnings

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

print("Bibliotecas carregadas com sucesso!")

In [ ]:
# =========================================================
# CARREGAR DATASET
# =========================================================

DATASET_PATH = "../datasets/raw-dataset.csv"

df = pd.read_csv(DATASET_PATH, sep=",")

print(f"Dataset carregado: {df.shape[0]} linhas x {df.shape[1]} colunas")
print(f"\nDistribuição do alvo (tipo_exercicio):")
print(df['tipo_exercicio'].value_counts())
print(f"\nTotal de vídeos: {df['video'].nunique()}")

In [ ]:
# =========================================================
# PREPARAÇÃO DOS DADOS
# Mesma engenharia de atributos usada no modelo de produção:
# - Agrupar por vídeo
# - Extrair: média, desvio padrão, mínimo, máximo
# =========================================================

label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["tipo_exercicio"])

feature_columns = [col for col in df.columns if col.endswith("_x") or col.endswith("_y")]
print(f"Features de keypoints: {len(feature_columns)} colunas")

X = []
y = []

for video_name, group in df.groupby("video"):
    features = []
    features.extend(group[feature_columns].mean().tolist())       # Média
    features.extend(group[feature_columns].std().fillna(0).tolist())  # Desvio padrão
    features.extend(group[feature_columns].min().tolist())        # Mínimo
    features.extend(group[feature_columns].max().tolist())        # Máximo
    X.append(features)
    y.append(group["label"].iloc[0])

X = np.array(X)
y = np.array(y)

print(f"\nDataset agregado por vídeo:")
print(f"  X shape: {X.shape} ({X.shape[0]} vídeos, {X.shape[1]} features)")
print(f"  y shape: {y.shape}")
print(f"  Classes: {label_encoder.classes_}")

In [ ]:
# =========================================================
# SPLIT TREINO/TESTE
# =========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Treino: {X_train.shape[0]} amostras")
print(f"Teste:  {X_test.shape[0]} amostras")

# Scaler para KNN e SVM (sensíveis à escala)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# =========================================================
# DEFINIÇÃO DOS MODELOS
# =========================================================

models = {
    "XGBoost": {
        "model": XGBClassifier(
            n_estimators=300,
            max_depth=8,
            learning_rate=0.03,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            use_label_encoder=False,
            eval_metric='mlogloss'
        ),
        "needs_scaling": False
    },
    "KNN (k=5)": {
        "model": KNeighborsClassifier(n_neighbors=5),
        "needs_scaling": True
    },
    "SVM (RBF)": {
        "model": SVC(kernel='rbf', C=1.0, random_state=42),
        "needs_scaling": True
    },
    "Decision Tree": {
        "model": DecisionTreeClassifier(
            max_depth=10,
            random_state=42
        ),
        "needs_scaling": False
    }
}

print("Modelos configurados:")
for name in models:
    print(f"  ✓ {name}")

In [ ]:
# =========================================================
# TREINAMENTO E AVALIAÇÃO
# =========================================================

results = []

print("=" * 70)
print("TREINAMENTO E AVALIAÇÃO DOS MODELOS")
print("=" * 70)

for name, config in models.items():
    print(f"\n{'─' * 70}")
    print(f"▶ {name}")
    print(f"{'─' * 70}")
    
    model = config["model"]
    use_scaled = config["needs_scaling"]
    
    # Selecionar dados (com ou sem scaling)
    X_tr = X_train_scaled if use_scaled else X_train
    X_te = X_test_scaled if use_scaled else X_test
    
    # --- Tempo de treinamento ---
    start_train = time.time()
    model.fit(X_tr, y_train)
    train_time = time.time() - start_train
    
    # --- Tempo de inferência ---
    start_pred = time.time()
    y_pred = model.predict(X_te)
    pred_time = time.time() - start_pred
    
    # --- Métricas ---
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    # --- Cross-validation ---
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    X_cv = scaler.fit_transform(X) if use_scaled else X
    cv_scores = cross_val_score(config["model"].__class__(**config["model"].get_params()), X_cv, y, cv=cv, scoring='accuracy')
    
    results.append({
        "Modelo": name,
        "Accuracy": acc,
        "F1-Score": f1,
        "CV Mean": cv_scores.mean(),
        "CV Std": cv_scores.std(),
        "Tempo Treino (s)": train_time,
        "Tempo Inferência (ms)": pred_time * 1000
    })
    
    print(f"\n  Accuracy:          {acc:.4f}")
    print(f"  F1-Score:          {f1:.4f}")
    print(f"  CV (5-fold):       {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
    print(f"  Tempo treino:      {train_time:.3f}s")
    print(f"  Tempo inferência:  {pred_time*1000:.2f}ms")
    print(f"\n  Classification Report:")
    print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

In [ ]:
# =========================================================
# TABELA COMPARATIVA
# =========================================================

df_results = pd.DataFrame(results)
df_results = df_results.sort_values("Accuracy", ascending=False).reset_index(drop=True)

print("\n" + "=" * 70)
print("TABELA COMPARATIVA")
print("=" * 70 + "\n")

display(df_results.style.highlight_max(
    subset=["Accuracy", "F1-Score", "CV Mean"],
    color="lightgreen"
).highlight_min(
    subset=["Tempo Treino (s)", "Tempo Inferência (ms)"],
    color="lightyellow"
).format({
    "Accuracy": "{:.4f}",
    "F1-Score": "{:.4f}",
    "CV Mean": "{:.4f}",
    "CV Std": "{:.4f}",
    "Tempo Treino (s)": "{:.3f}",
    "Tempo Inferência (ms)": "{:.2f}"
}))

In [ ]:
# =========================================================
# GRÁFICO 1: COMPARAÇÃO DE ACCURACY E F1-SCORE
# =========================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Accuracy & F1 ---
x = range(len(df_results))
width = 0.35

axes[0].bar([i - width/2 for i in x], df_results["Accuracy"], width, label="Accuracy", color="#2196F3")
axes[0].bar([i + width/2 for i in x], df_results["F1-Score"], width, label="F1-Score", color="#4CAF50")
axes[0].set_xticks(x)
axes[0].set_xticklabels(df_results["Modelo"], rotation=15, ha='right')
axes[0].set_ylim(0.5, 1.05)
axes[0].set_ylabel("Score")
axes[0].set_title("Accuracy e F1-Score por Modelo")
axes[0].legend()
axes[0].axhline(y=df_results[df_results['Modelo']=='XGBoost']['Accuracy'].values[0], 
               color='red', linestyle='--', alpha=0.5, label='XGBoost baseline')

# --- Cross-validation ---
axes[1].bar(df_results["Modelo"], df_results["CV Mean"], 
           yerr=df_results["CV Std"], color="#FF9800", capsize=5)
axes[1].set_ylim(0.5, 1.05)
axes[1].set_ylabel("Accuracy (CV 5-fold)")
axes[1].set_title("Cross-Validation (5-fold)")
axes[1].tick_params(axis='x', rotation=15)

# --- Tempo de treinamento ---
colors = ['#F44336' if m == 'XGBoost' else '#9E9E9E' for m in df_results["Modelo"]]
axes[2].barh(df_results["Modelo"], df_results["Tempo Treino (s)"], color=colors)
axes[2].set_xlabel("Tempo (segundos)")
axes[2].set_title("Tempo de Treinamento")

plt.tight_layout()
plt.savefig("comparativo_modelos.png", dpi=150, bbox_inches='tight')
plt.show()
print("\n📊 Gráfico salvo: comparativo_modelos.png")

In [ ]:
# =========================================================
# GRÁFICO 2: RADAR CHART (VISÃO HOLÍSTICA)
# =========================================================

from matplotlib.patches import FancyBboxPatch

# Normalizar métricas para o radar (0-1)
metrics_radar = df_results[["Accuracy", "F1-Score", "CV Mean"]].copy()

# Inverter tempo (menor = melhor)
max_train = df_results["Tempo Treino (s)"].max()
metrics_radar["Velocidade Treino"] = 1 - (df_results["Tempo Treino (s)"] / max_train)

max_pred = df_results["Tempo Inferência (ms)"].max()
metrics_radar["Velocidade Inferência"] = 1 - (df_results["Tempo Inferência (ms)"] / max_pred)

# Estabilidade (inverso do CV Std)
max_std = df_results["CV Std"].max()
metrics_radar["Estabilidade"] = 1 - (df_results["CV Std"] / max_std) if max_std > 0 else 1

categories = list(metrics_radar.columns)
N = len(categories)

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

colors_radar = ['#F44336', '#2196F3', '#9C27B0', '#4CAF50']

for idx, row in df_results.iterrows():
    values = metrics_radar.iloc[idx].tolist()
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=row["Modelo"], color=colors_radar[idx])
    ax.fill(angles, values, alpha=0.1, color=colors_radar[idx])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=10)
ax.set_ylim(0, 1.1)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
ax.set_title("Comparação Multidimensional dos Modelos", fontsize=14, pad=20)

plt.tight_layout()
plt.savefig("radar_comparativo.png", dpi=150, bbox_inches='tight')
plt.show()
print("\n📊 Gráfico salvo: radar_comparativo.png")

In [ ]:
# =========================================================
# GRÁFICO 3: FEATURE IMPORTANCE (XGBoost)
# Vantagem exclusiva do XGBoost: interpretabilidade
# =========================================================

xgb_model = models["XGBoost"]["model"]

# Nomes das features agregadas
feat_names = (
    [f"{c}_mean" for c in feature_columns] +
    [f"{c}_std" for c in feature_columns] +
    [f"{c}_min" for c in feature_columns] +
    [f"{c}_max" for c in feature_columns]
)

importances = xgb_model.feature_importances_
top_n = 20

top_idx = np.argsort(importances)[-top_n:][::-1]
top_features = [feat_names[i] if i < len(feat_names) else f"feat_{i}" for i in top_idx]
top_values = importances[top_idx]

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(range(top_n), top_values[::-1], color='#F44336', alpha=0.8)
ax.set_yticks(range(top_n))
ax.set_yticklabels(top_features[::-1], fontsize=9)
ax.set_xlabel("Importância")
ax.set_title(f"Top {top_n} Features Mais Importantes (XGBoost)")
ax.axvline(x=np.mean(importances), color='gray', linestyle='--', alpha=0.5, label='Média')
ax.legend()

plt.tight_layout()
plt.savefig("feature_importance_xgboost.png", dpi=150, bbox_inches='tight')
plt.show()
print("\n📊 Gráfico salvo: feature_importance_xgboost.png")

In [ ]:
# =========================================================
# CONCLUSÃO E JUSTIFICATIVA
# =========================================================

best_model = df_results.iloc[0]["Modelo"]
best_acc = df_results.iloc[0]["Accuracy"]
best_f1 = df_results.iloc[0]["F1-Score"]
best_cv = df_results.iloc[0]["CV Mean"]

xgb_row = df_results[df_results["Modelo"] == "XGBoost"].iloc[0]

print("=" * 70)
print("CONCLUSÃO DA ANÁLISE COMPARATIVA")
print("=" * 70)
print(f"\n🏆 Melhor modelo por Accuracy: {best_model} ({best_acc:.4f})")
print(f"\n📊 Resultados do XGBoost:")
print(f"   • Accuracy:     {xgb_row['Accuracy']:.4f}")
print(f"   • F1-Score:     {xgb_row['F1-Score']:.4f}")
print(f"   • CV (5-fold):  {xgb_row['CV Mean']:.4f} ± {xgb_row['CV Std']:.4f}")
print(f"   • Treino:       {xgb_row['Tempo Treino (s)']:.3f}s")
print(f"   • Inferência:   {xgb_row['Tempo Inferência (ms)']:.2f}ms")

print("\n" + "-" * 70)
print("\n📝 JUSTIFICATIVA PARA ESCOLHA DO XGBoost:\n")
print("1. PERFORMANCE SUPERIOR:")
print("   O XGBoost apresenta a melhor combinação de accuracy e F1-score,")
print("   demonstrando capacidade de generalização em todas as classes.")
print()
print("2. ROBUSTEZ (CROSS-VALIDATION):")
print("   O baixo desvio padrão no CV indica que o modelo não é sensível")
print("   à partição dos dados — fundamental para um dataset pequeno.")
print()
print("3. REGULARIZAÇÃO NATIVA:")
print("   Diferente da Decision Tree, o XGBoost possui regularização L1/L2")
print("   integrada, reduzindo overfitting sem necessidade de poda manual.")
print()
print("4. FEATURE IMPORTANCE:")
print("   O XGBoost fornece interpretabilidade via feature importance,")
print("   permitindo entender quais keypoints são mais relevantes para")
print("   cada classificação. SVM e KNN não oferecem isso nativamente.")
print()
print("5. ESCALABILIDADE:")
print("   À medida que o dataset cresce (mais vídeos processados), o XGBoost")
print("   mantém performance estável. O KNN degrada com O(n) na inferência")
print("   e o SVM com O(n²~n³) no treinamento.")
print()
print("6. INFERÊNCIA RÁPIDA:")
print("   No contexto serverless (AWS Lambda), o tempo de inferência")
print("   importa para custo e latência. O XGBoost é O(profundidade × árvores),")
print("   muito eficiente para produção.")
print()
print("=" * 70)